# 05.10 - Support Vector Machines

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Support Vector Machines (SVM) find the hyperplane that best separates classes by maximizing the **margin** (distance to the nearest points, called support vectors). The **kernel trick** lets SVM handle nonlinear boundaries.

## 2. Why Does This Matter?

SVM is powerful for small-to-medium datasets, especially with clear margins. Understanding margins and kernels is foundational.

## 3. Prerequisites

- Phase 02 (Math - vectors, dot products)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain margin and support vectors
- Explain the kernel trick
- Use sklearn's SVM with linear and RBF kernels
- Understand the C parameter

## 5. Mental Model

SVM finds the hyperplane with the maximum margin:

- **Support vectors**: the closest points to the hyperplane.
- **Margin**: distance between hyperplane and support vectors.
- **C**: tradeoff between margin size and misclassification.
- **Kernel**: maps data to higher dimensions to find nonlinear boundaries.


## 6. Generate Data

Create a linearly separable dataset.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


## 7. Linear SVM

Train a linear SVM and visualize the decision boundary.


In [ ]:
model = SVC(kernel='linear', C=1.0)
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))
print(f"Linear SVM accuracy: {acc:.3f}")
print(f"Number of support vectors: {len(model.support_vectors_)}")

# Visualize decision boundary
def plot_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min()-1, X[:, 0].max()+1
    y_min, y_max = X[:, 1].min()-1, X[:, 1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[y==0, 0], X[y==0, 1], alpha=0.5, label="Class 0")
    plt.scatter(X[y==1, 0], X[y==1, 1], alpha=0.5, label="Class 1")
    plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1], s=100, facecolors='none', edgecolors='k', label="Support vectors")
    plt.title(title)
    plt.legend()

plt.figure(figsize=(6, 5))
plot_boundary(model, X_test, y_test, "Linear SVM")
plt.show()


## 8. Nonlinear Data and the Kernel Trick

For nonlinear data, the RBF kernel maps to higher dimensions.


In [ ]:
# Create nonlinear (circular) data
from sklearn.datasets import make_circles
Xc, yc = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=42)

# Linear kernel fails on circular data
lin = SVC(kernel='linear').fit(Xc_tr, yc_tr)
lin_acc = accuracy_score(yc_te, lin.predict(Xc_te))

# RBF kernel handles it
rbf = SVC(kernel='rbf').fit(Xc_tr, yc_tr)
rbf_acc = accuracy_score(yc_te, rbf.predict(Xc_te))

print(f"Linear kernel on circles: {lin_acc:.3f}")
print(f"RBF kernel on circles:    {rbf_acc:.3f}")
print("\nThe kernel trick handles nonlinear boundaries.")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plot_boundary(lin, Xc_te, yc_te, "Linear kernel")
plt.subplot(1, 2, 2)
plot_boundary(rbf, Xc_te, yc_te, "RBF kernel")
plt.show()


## 9. Effect of C

C controls the tradeoff between margin and misclassification.


In [ ]:
for C in [0.01, 0.1, 1.0, 10.0]:
    m = SVC(kernel='linear', C=C).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    n_sv = len(m.support_vectors_)
    print(f"C={C:5.2f}: accuracy={acc:.3f}, support vectors={n_sv}")
print("\nSmall C = wider margin, more support vectors; large C = tighter fit.")


## 10. Failure Case: Unscaled Features

SVM is sensitive to feature scale.


In [ ]:
from sklearn.preprocessing import StandardScaler
X_big = X.copy()
X_big[:, 1] *= 1000
Xb_tr, Xb_te, _, _ = train_test_split(X_big, y, test_size=0.3, random_state=42)

m_un = SVC(kernel='rbf').fit(Xb_tr, y_train)
acc_un = accuracy_score(y_test, m_un.predict(Xb_te))

sc = StandardScaler().fit(Xb_tr)
m_sc = SVC(kernel='rbf').fit(sc.transform(Xb_tr), y_train)
acc_sc = accuracy_score(y_test, m_sc.predict(sc.transform(Xb_te)))

print(f"Unscaled accuracy: {acc_un:.3f}")
print(f"Scaled accuracy:   {acc_sc:.3f}")
print("\nAlways scale features for SVM.")


## 11. Debugging: Common Errors

- **Not scaling**: SVM is scale-sensitive.
- **Wrong kernel**: linear on nonlinear data.
- **C too large**: overfitting.

## 12. Real-World Considerations

- SVM is good for small/medium datasets.
- Slow on large datasets.
- RBF kernel is a good default.

## 13. Common Mistakes

- Forgetting to scale.
- Using linear kernel on nonlinear data.

## 14. When NOT to Use

- Very large datasets.
- When you need probability estimates (use calibration).

## 15. Challenge

Compare SVM with different kernels on a dataset and report which performs best.


In [ ]:
# Challenge: compare kernels
for kernel in ['linear', 'poly', 'rbf', 'sigmoid']:
    m = SVC(kernel=kernel).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"kernel={kernel:8s}: accuracy={acc:.3f}")
print("\nRBF usually performs best on nonlinear data.")


## 16. Closed-Book Recall

Without looking back:

1. What is a support vector?
2. What is the margin?
3. What does the kernel trick do?
4. What does C control?

## 17. Teach-Back Questions

Explain to another person:

- How SVM finds the best hyperplane.
- Why the kernel trick is powerful.

## 18. Summary

You trained SVMs with linear and RBF kernels, visualized decision boundaries, and explored the C parameter and scaling.

## 19. Further Experiment

- Tune gamma for the RBF kernel.
- Use SVR for regression.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
